# I04a: Pretrained Foundation Models in Computer Vision

**Course:** COMPSCI 714 - AI Architecture and Design  
**Lecture:** 8-2 (Pretrained Foundational Model II)  

This notebook covers pretrained foundation models in computer vision as presented in
COMPSCI 714 Lecture 8-2. We explore the pre-training and fine-tuning paradigm,
self-supervised learning tasks, and key architectures including ResNet, Vision Transformer (ViT),
BEiT, and CLIP.

All implementations use PyTorch and are designed to run in Google Colab.

## Table of Contents

1. [Pre-training and Fine-tuning Paradigm](#1-pre-training-and-fine-tuning-paradigm)
2. [Self-supervised Learning Tasks in CV](#2-self-supervised-learning-tasks-in-cv)
   - 2.1 [Image Colorization](#21-image-colorization)
   - 2.2 [Contrastive Learning (SimCLR)](#22-contrastive-learning-simclr)
   - 2.3 [Masked Image Modeling](#23-masked-image-modeling)
   - 2.4 [Rotation Prediction](#24-rotation-prediction)
   - 2.5 [Jigsaw Puzzle](#25-jigsaw-puzzle)
3. [ResNet: Residual Networks](#3-resnet-residual-networks)
4. [Vision Transformer (ViT)](#4-vision-transformer-vit)
5. [BEiT: BERT Pre-training for Images](#5-beit-bert-pre-training-for-images)
6. [CLIP: Contrastive Language-Image Pre-training](#6-clip-contrastive-language-image-pre-training)
7. [Practical Example: Using Pretrained Models](#7-practical-example-using-pretrained-models)
8. [Summary](#8-summary)

---
## Setup

Install and import required libraries.

In [ ]:
# Setup - run this cell first
# !pip install torch torchvision matplotlib numpy pillow  # Uncomment if needed in Colab

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from copy import deepcopy
import math

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')
print(f'Torchvision version: {torchvision.__version__}')
print('Setup complete!')

---
## 1. Pre-training and Fine-tuning Paradigm

### The Core Idea

The **pre-training and fine-tuning paradigm** is the dominant approach in modern deep learning:

1. **Pre-training:** Train a large model on massive unlabeled data using **self-supervised learning**.
   The model learns general, transferable representations (features) without human annotations.

2. **Fine-tuning:** Adapt the pre-trained model to a specific downstream task using a small amount
   of labeled data. Only minor adjustments are needed since the model already understands
   general visual concepts.

### Why This Works

- **Labeled data is expensive** — annotation requires human effort
- **Unlabeled data is abundant** — billions of images available on the internet
- **General features transfer** — edges, textures, shapes, and object parts are useful across tasks

### Foundation / Base Models

A **foundation model** (also called a **base model**) is a large model pre-trained on broad data
that can be adapted to many downstream tasks:

| Property | Description |
|----------|-------------|
| Scale | Trained on massive datasets (millions/billions of samples) |
| Self-supervised | No manual labels needed for pre-training |
| Transferable | Features generalize across tasks and domains |
| Adaptable | Fine-tuned with small task-specific data |

**Examples:** ResNet (ImageNet pre-trained), ViT, BEiT, CLIP, DINO, MAE

```
Pre-training Phase:              Fine-tuning Phase:
┌─────────────────┐              ┌─────────────────┐
│  Large Unlabeled │              │  Small Labeled   │
│     Dataset      │              │    Dataset       │
└────────┬────────┘              └────────┬────────┘
         │                                │
         ▼                                ▼
┌─────────────────┐              ┌─────────────────┐
│  Self-supervised │              │  Supervised      │
│  Learning Task   │              │  Task-specific   │
└────────┬────────┘              └────────┬────────┘
         │                                │
         ▼                                ▼
┌─────────────────┐              ┌─────────────────┐
│  Foundation      │──transfer──▶│  Fine-tuned      │
│  Model           │              │  Model           │
└─────────────────┘              └─────────────────┘
```

In [ ]:
# Demonstration: Pre-training vs Fine-tuning concept
# We'll show how a pre-trained feature extractor helps with limited data

class SimpleEncoder(nn.Module):
    """A simple encoder that learns general features."""
    def __init__(self, input_dim=784, hidden_dim=256, feature_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, feature_dim),
            nn.ReLU()
        )
    
    def forward(self, x):
        return self.encoder(x)

class FineTuneClassifier(nn.Module):
    """Classifier head added on top of pre-trained encoder."""
    def __init__(self, encoder, feature_dim=64, num_classes=10):
        super().__init__()
        self.encoder = encoder  # Pre-trained, can freeze or fine-tune
        self.classifier = nn.Linear(feature_dim, num_classes)
    
    def forward(self, x):
        features = self.encoder(x)
        return self.classifier(features)

# Simulate pre-training: encoder learns general features
pretrained_encoder = SimpleEncoder()

# Simulate fine-tuning: add task-specific head
model = FineTuneClassifier(pretrained_encoder, num_classes=10)

# Option 1: Freeze encoder (linear probing)
for param in model.encoder.parameters():
    param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters (fine-tuning head only): {trainable_params:,}')
print(f'Frozen parameters: {total_params - trainable_params:,}')
print(f'\nOnly {trainable_params/total_params*100:.1f}% of parameters need training!')

---
## 2. Self-supervised Learning Tasks in CV

Self-supervised learning (SSL) creates **pretext tasks** from the data itself — no human labels needed.
The model learns useful representations by solving these tasks.

### Key Principle
Design a task where the **supervision signal comes from the data structure itself**:
- Hide part of the input → predict the hidden part
- Transform the input → predict the transformation
- Create pairs → learn which pairs match

The learned features transfer well to downstream tasks (classification, detection, segmentation).

### 2.1 Image Colorization

**Task:** Given a grayscale image, predict the color channels.

**Intuition:** To colorize correctly, the model must understand object semantics
(sky is blue, grass is green, etc.).

**Setup:**
- Input: Grayscale image (L channel in Lab color space)
- Target: Color channels (ab channels)
- Loss: L1 or L2 (MSE) reconstruction loss

$$\mathcal{L}_{\text{color}} = \|\hat{y}_{ab} - y_{ab}\|_1$$

The model learns to associate visual patterns with colors, developing rich semantic features.

In [ ]:
# 2.1 Image Colorization - Simplified Demo

class ColorizationNet(nn.Module):
    """Simple colorization network.
    Input: grayscale image (1 channel)
    Output: predicted color channels (2 channels for ab in Lab space)
    """
    def __init__(self):
        super().__init__()
        # Encoder: learns features from grayscale
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.ReLU(),
        )
        # Decoder: predicts color from features
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 2, 3, padding=1),  # 2 output channels (ab)
            nn.Tanh()  # ab values normalized to [-1, 1]
        )
    
    def forward(self, x):
        features = self.encoder(x)
        color_pred = self.decoder(features)
        return color_pred

# Create model and demonstrate
colorization_model = ColorizationNet()

# Simulate: grayscale input (batch=1, channels=1, 32x32)
grayscale_input = torch.randn(1, 1, 32, 32)
predicted_ab = colorization_model(grayscale_input)

print(f'Input (grayscale): {grayscale_input.shape}')  # [1, 1, 32, 32]
print(f'Output (predicted ab): {predicted_ab.shape}')  # [1, 2, 32, 32]

# Loss: L1 reconstruction
target_ab = torch.randn(1, 2, 32, 32)  # Ground truth color channels
loss_l1 = F.l1_loss(predicted_ab, target_ab)
loss_l2 = F.mse_loss(predicted_ab, target_ab)
print(f'\nL1 Loss: {loss_l1.item():.4f}')
print(f'L2 (MSE) Loss: {loss_l2.item():.4f}')
print('\nAfter pre-training, the encoder learns semantic features useful for downstream tasks.')

### 2.2 Contrastive Learning (SimCLR)

**Task:** Learn representations where similar images are close and dissimilar images are far apart.

**SimCLR Framework (Chen et al., 2020):**
1. Take an image $x$
2. Apply two random augmentations → get **positive pair** $(x_i, x_j)$
3. Other images in the batch form **negative pairs**
4. Train encoder to pull positives together, push negatives apart

**InfoNCE Loss (NT-Xent):**

$$\mathcal{L}_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)}$$

Where:
- $\text{sim}(z_i, z_j) = \frac{z_i^T z_j}{\|z_i\| \|z_j\|}$ (cosine similarity)
- $\tau$ is a temperature parameter
- $N$ is the batch size

```
Image x ──┬── Augment 1 ──▶ x_i ──▶ Encoder ──▶ z_i ──┐
           │                                              │ Pull together
           └── Augment 2 ──▶ x_j ──▶ Encoder ──▶ z_j ──┘
                                                          │ Push apart
Other images ──▶ Encoder ──▶ z_k ────────────────────────┘
```

In [ ]:
# 2.2 Contrastive Learning (SimCLR-style) Implementation

class SimCLREncoder(nn.Module):
    """SimCLR: Encoder + Projection Head."""
    def __init__(self, feature_dim=128):
        super().__init__()
        # Backbone encoder (simplified CNN)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        # Projection head (maps to contrastive space)
        self.projection = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, feature_dim)
        )
    
    def forward(self, x):
        h = self.encoder(x)       # Representation
        z = self.projection(h)    # Projection for contrastive loss
        return F.normalize(z, dim=1)  # L2 normalize


def info_nce_loss(z_i, z_j, temperature=0.5):
    """Compute InfoNCE (NT-Xent) loss for a batch of positive pairs.
    
    Args:
        z_i: embeddings from augmentation 1 [batch_size, feature_dim]
        z_j: embeddings from augmentation 2 [batch_size, feature_dim]
        temperature: scaling factor
    """
    batch_size = z_i.shape[0]
    
    # Concatenate both views
    z = torch.cat([z_i, z_j], dim=0)  # [2*batch_size, feature_dim]
    
    # Compute similarity matrix
    sim_matrix = torch.mm(z, z.t()) / temperature  # [2N, 2N]
    
    # Create labels: positive pairs are (i, i+N) and (i+N, i)
    labels = torch.cat([torch.arange(batch_size) + batch_size,
                        torch.arange(batch_size)]).to(z.device)
    
    # Mask out self-similarity (diagonal)
    mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z.device)
    sim_matrix.masked_fill_(mask, -1e9)
    
    # Cross-entropy loss
    loss = F.cross_entropy(sim_matrix, labels)
    return loss


# Demo: SimCLR forward pass
simclr_model = SimCLREncoder(feature_dim=128)

# Simulate a batch of augmented pairs
batch_size = 8
x_aug1 = torch.randn(batch_size, 3, 32, 32)  # Augmentation 1
x_aug2 = torch.randn(batch_size, 3, 32, 32)  # Augmentation 2

z_i = simclr_model(x_aug1)
z_j = simclr_model(x_aug2)

loss = info_nce_loss(z_i, z_j, temperature=0.5)
print(f'Batch size: {batch_size}')
print(f'Embedding shape: {z_i.shape}')
print(f'InfoNCE Loss: {loss.item():.4f}')
print(f'\nPositive pairs: {batch_size} (same image, different augmentations)')
print(f'Negative pairs per sample: {2*batch_size - 2} (all other images in batch)')

### 2.3 Masked Image Modeling

**Task:** Mask random patches of an image and predict the missing content.

**Inspired by BERT's masked language modeling**, applied to images:
1. Split image into patches (e.g., 16×16)
2. Randomly mask a large portion (typically **75%** of patches)
3. Encode visible patches
4. Predict/reconstruct the masked patches

**Key insight:** High masking ratio (75%) forces the model to learn holistic understanding
rather than just interpolating from neighbors.

**Used in:** MAE (Masked Autoencoders), BEiT, SimMIM

```
Original Image:          Masked (75%):           Reconstructed:
┌──┬──┬──┬──┐           ┌──┬──┬──┬──┐           ┌──┬──┬──┬──┐
│P1│P2│P3│P4│           │██│P2│██│██│           │P1│P2│P3│P4│
├──┼──┼──┼──┤           ├──┼──┼──┼──┤           ├──┼──┼──┼──┤
│P5│P6│P7│P8│    ──▶    │██│██│P7│██│    ──▶    │P5│P6│P7│P8│
├──┼──┼──┼──┤           ├──┼──┼──┼──┤           ├──┼──┼──┼──┤
│P9│10│11│12│           │██│██│██│12│           │P9│10│11│12│
└──┴──┴──┴──┘           └──┴──┴──┴──┘           └──┴──┴──┴──┘
```

In [ ]:
# 2.3 Masked Image Modeling (MAE-style)

class MaskedImageModeling(nn.Module):
    """Simplified Masked Autoencoder for images."""
    def __init__(self, image_size=32, patch_size=8, embed_dim=128, mask_ratio=0.75):
        super().__init__()
        self.patch_size = patch_size
        self.mask_ratio = mask_ratio
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim = 3 * patch_size * patch_size  # RGB patches
        
        # Patch embedding
        self.patch_embed = nn.Linear(patch_dim, embed_dim)
        
        # Simple transformer encoder (1 layer for demo)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=256, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # Decoder: reconstruct patches
        self.decoder = nn.Linear(embed_dim, patch_dim)
    
    def patchify(self, images):
        """Split images into patches. [B, 3, H, W] -> [B, num_patches, patch_dim]"""
        B, C, H, W = images.shape
        p = self.patch_size
        patches = images.unfold(2, p, p).unfold(3, p, p)  # [B, C, H/p, W/p, p, p]
        patches = patches.contiguous().view(B, C, -1, p, p)
        patches = patches.permute(0, 2, 1, 3, 4).contiguous()  # [B, num_patches, C, p, p]
        patches = patches.view(B, -1, C * p * p)  # [B, num_patches, patch_dim]
        return patches
    
    def random_mask(self, num_patches, batch_size):
        """Generate random mask. Returns indices of visible and masked patches."""
        num_masked = int(num_patches * self.mask_ratio)
        
        # Random permutation for each sample
        noise = torch.rand(batch_size, num_patches)
        ids_shuffle = torch.argsort(noise, dim=1)
        
        ids_visible = ids_shuffle[:, :num_patches - num_masked]
        ids_masked = ids_shuffle[:, num_patches - num_masked:]
        return ids_visible, ids_masked
    
    def forward(self, images):
        B = images.shape[0]
        patches = self.patchify(images)  # [B, N, patch_dim]
        
        # Random masking
        ids_visible, ids_masked = self.random_mask(self.num_patches, B)
        
        # Embed visible patches only
        visible_patches = torch.gather(
            patches, 1, ids_visible.unsqueeze(-1).expand(-1, -1, patches.shape[-1])
        )
        embedded = self.patch_embed(visible_patches)
        
        # Encode
        encoded = self.encoder(embedded)
        
        # Decode (predict masked patches)
        reconstructed = self.decoder(encoded)
        
        return reconstructed, ids_visible, ids_masked, patches


# Demo
mae_model = MaskedImageModeling(image_size=32, patch_size=8, mask_ratio=0.75)
dummy_images = torch.randn(4, 3, 32, 32)

recon, vis_ids, mask_ids, original_patches = mae_model(dummy_images)

num_patches = (32 // 8) ** 2
num_visible = vis_ids.shape[1]
num_masked = mask_ids.shape[1]

print(f'Image size: 32x32, Patch size: 8x8')
print(f'Total patches: {num_patches}')
print(f'Visible patches: {num_visible} ({num_visible/num_patches*100:.0f}%)')
print(f'Masked patches: {num_masked} ({num_masked/num_patches*100:.0f}%)')
print(f'Reconstructed shape: {recon.shape}')

In [ ]:
# Visualize masking process
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Create a sample image (checkerboard pattern for visibility)
sample_img = torch.zeros(3, 32, 32)
for i in range(4):
    for j in range(4):
        color = torch.rand(3) * 0.5 + 0.3
        sample_img[:, i*8:(i+1)*8, j*8:(j+1)*8] = color.unsqueeze(1).unsqueeze(2)

# Show original
axes[0].imshow(sample_img.permute(1, 2, 0).numpy())
axes[0].set_title('Original (16 patches)')
axes[0].axis('off')

# Show masked version (75% masked)
masked_img = sample_img.clone()
mask_indices = torch.randperm(16)[:12]  # Mask 12 of 16 patches (75%)
for idx in mask_indices:
    row, col = idx // 4, idx % 4
    masked_img[:, row*8:(row+1)*8, col*8:(col+1)*8] = 0.5  # Gray = masked

axes[1].imshow(masked_img.permute(1, 2, 0).numpy())
axes[1].set_title('Masked (75% patches hidden)')
axes[1].axis('off')

# Show reconstruction target
axes[2].imshow(sample_img.permute(1, 2, 0).numpy())
axes[2].set_title('Target: Reconstruct masked patches')
axes[2].axis('off')

plt.suptitle('Masked Image Modeling: Pretext Task', fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Rotation Prediction

**Task:** Rotate an image by 0°, 90°, 180°, or 270° and predict which rotation was applied.

**Intuition:** To predict rotation, the model must understand:
- Object orientation (cars don't fly upside down)
- Scene layout (sky is above, ground is below)
- Gravity and spatial relationships

**Setup:**
- Input: Rotated image
- Output: 4-class classification (0°, 90°, 180°, 270°)
- Loss: Cross-entropy

This is one of the simplest yet effective SSL pretext tasks (Gidaris et al., 2018).

In [ ]:
# 2.4 Rotation Prediction

class RotationPrediction(nn.Module):
    """Predict which rotation (0, 90, 180, 270 degrees) was applied."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        self.classifier = nn.Linear(128, 4)  # 4 rotation classes
    
    def forward(self, x):
        features = self.features(x)
        return self.classifier(features)


def create_rotation_batch(images):
    """Create rotated versions of images with labels."""
    rotated_images = []
    labels = []
    
    for img in images:
        # Randomly choose a rotation
        rotation = torch.randint(0, 4, (1,)).item()
        rotated = torch.rot90(img, k=rotation, dims=[1, 2])
        rotated_images.append(rotated)
        labels.append(rotation)
    
    return torch.stack(rotated_images), torch.tensor(labels)


# Demo
rot_model = RotationPrediction()

# Create a batch of images and rotate them
original_images = torch.randn(8, 3, 32, 32)
rotated_batch, rotation_labels = create_rotation_batch(original_images)

# Forward pass
predictions = rot_model(rotated_batch)
loss = F.cross_entropy(predictions, rotation_labels)

print(f'Input shape: {rotated_batch.shape}')
print(f'Rotation labels: {rotation_labels.tolist()}')
print(f'  (0=0°, 1=90°, 2=180°, 3=270°)')
print(f'Predictions shape: {predictions.shape}')
print(f'Cross-entropy loss: {loss.item():.4f}')

In [ ]:
# Visualize rotation prediction task
fig, axes = plt.subplots(1, 4, figsize=(12, 3))

# Create a simple recognizable pattern
sample = torch.zeros(3, 32, 32)
# Draw an 'F' shape so rotation is visible
sample[0, 4:28, 4:8] = 1.0    # Vertical bar (red)
sample[0, 4:8, 4:20] = 1.0    # Top horizontal
sample[0, 14:18, 4:16] = 1.0  # Middle horizontal
sample[1, 4:28, 4:8] = 0.3    # Add some green

rotation_names = ['0° (Original)', '90° Rotation', '180° Rotation', '270° Rotation']

for i in range(4):
    rotated = torch.rot90(sample, k=i, dims=[1, 2])
    axes[i].imshow(rotated.permute(1, 2, 0).numpy())
    axes[i].set_title(f'Class {i}: {rotation_names[i]}')
    axes[i].axis('off')

plt.suptitle('Rotation Prediction: 4-class Classification Task', fontsize=13)
plt.tight_layout()
plt.show()

### 2.5 Jigsaw Puzzle

**Task:** Split an image into a grid of patches, shuffle them, and predict the correct order.

**Intuition:** To solve a jigsaw puzzle, the model must understand:
- Spatial relationships between parts
- Object structure and continuity
- Scene composition

**Setup (Noroozi & Favaro, 2016):**
1. Divide image into a 3×3 grid (9 patches)
2. Shuffle patches according to a predefined permutation
3. Predict which permutation was used

**Note:** With 9 patches, there are 9! = 362,880 possible permutations.
In practice, a subset of ~1000 maximally different permutations is used.

In [ ]:
# 2.5 Jigsaw Puzzle Solving

class JigsawPuzzle(nn.Module):
    """Predict the permutation used to shuffle image patches."""
    def __init__(self, num_permutations=100):
        super().__init__()
        # Process each patch independently
        self.patch_encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        # Combine all patch features and predict permutation
        self.classifier = nn.Sequential(
            nn.Linear(32 * 9, 256),  # 9 patches, each 32-dim
            nn.ReLU(),
            nn.Linear(256, num_permutations)  # Predict which permutation
        )
    
    def forward(self, patches):
        """patches: [batch, 9, 3, patch_h, patch_w]"""
        B, N, C, H, W = patches.shape
        # Encode each patch
        patch_features = []
        for i in range(N):
            feat = self.patch_encoder(patches[:, i])  # [B, 32]
            patch_features.append(feat)
        
        # Concatenate all patch features
        combined = torch.cat(patch_features, dim=1)  # [B, 32*9]
        return self.classifier(combined)


def create_jigsaw_batch(images, patch_size=10, num_permutations=100):
    """Create jigsaw puzzles from images."""
    B, C, H, W = images.shape
    grid_size = 3  # 3x3 grid
    p = H // grid_size
    
    # Pre-define permutations (subset)
    torch.manual_seed(0)
    permutations = [torch.randperm(9) for _ in range(num_permutations)]
    
    shuffled_patches = []
    labels = []
    
    for img in images:
        # Extract 3x3 patches
        patches = []
        for i in range(grid_size):
            for j in range(grid_size):
                patch = img[:, i*p:(i+1)*p, j*p:(j+1)*p]
                patches.append(patch)
        patches = torch.stack(patches)  # [9, C, p, p]
        
        # Choose random permutation
        perm_idx = torch.randint(0, num_permutations, (1,)).item()
        perm = permutations[perm_idx]
        shuffled = patches[perm]  # Apply permutation
        
        shuffled_patches.append(shuffled)
        labels.append(perm_idx)
    
    return torch.stack(shuffled_patches), torch.tensor(labels)


# Demo
jigsaw_model = JigsawPuzzle(num_permutations=100)
images = torch.randn(4, 3, 30, 30)  # 30x30 so patches are 10x10

shuffled, perm_labels = create_jigsaw_batch(images, num_permutations=100)
predictions = jigsaw_model(shuffled)
loss = F.cross_entropy(predictions, perm_labels)

print(f'Image size: 30x30, Grid: 3x3, Patch size: 10x10')
print(f'Shuffled patches shape: {shuffled.shape}')  # [4, 9, 3, 10, 10]
print(f'Permutation labels: {perm_labels.tolist()}')
print(f'Number of permutation classes: 100')
print(f'Cross-entropy loss: {loss.item():.4f}')

---
## 3. ResNet: Residual Networks

### The Degradation Problem

Deeper networks should perform at least as well as shallower ones (extra layers could learn identity).
But in practice, **very deep plain networks perform worse** — not due to overfitting, but due to
optimization difficulty.

### Solution: Skip Connections (He et al., 2015)

Instead of learning $H(x)$ directly, learn the **residual** $F(x) = H(x) - x$:

$$H(x) = F(x) + x$$

**Why this works:**
- If identity is optimal, it's easier to push $F(x) → 0$ than to learn $H(x) = x$
- Gradients flow directly through skip connections (no vanishing gradient)
- Enables training of very deep networks (100+ layers)

### Residual Block

```
x ──────────────────────────────┐
│                                │ (skip/shortcut connection)
▼                                │
┌─────────────┐                  │
│  Conv 3×3   │                  │
│  BatchNorm  │                  │
│  ReLU       │                  │
├─────────────┤                  │
│  Conv 3×3   │                  │
│  BatchNorm  │                  │
└──────┬──────┘                  │
       │                         │
       ▼                         ▼
       ┌─────────────────────────┐
       │      F(x) + x          │  ← Element-wise addition
       └────────────┬────────────┘
                    │
                    ▼
                  ReLU
```

### ResNet Variants

| Model | Layers | Parameters | Top-1 Accuracy (ImageNet) |
|-------|--------|------------|---------------------------|
| ResNet-18 | 18 | 11.7M | 69.8% |
| ResNet-34 | 34 | 21.8M | 73.3% |
| ResNet-50 | 50 | 25.6M | 76.1% |
| ResNet-101 | 101 | 44.5M | 77.4% |
| ResNet-152 | 152 | 60.2M | 78.3% |

In [ ]:
# 3. ResNet: Building Residual Blocks from Scratch

class ResidualBlock(nn.Module):
    """Basic residual block: H(x) = F(x) + x"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Shortcut connection (identity or projection)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            # Need 1x1 conv to match dimensions
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        # F(x): the residual function
        residual = self.conv1(x)
        residual = self.bn1(residual)
        residual = F.relu(residual)
        residual = self.conv2(residual)
        residual = self.bn2(residual)
        
        # H(x) = F(x) + x
        out = residual + self.shortcut(x)  # Skip connection!
        out = F.relu(out)
        return out


class SimpleResNet(nn.Module):
    """Simplified ResNet for demonstration."""
    def __init__(self, num_classes=10):
        super().__init__()
        # Initial convolution
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )
        # Residual layers
        self.layer1 = self._make_layer(64, 64, num_blocks=2, stride=1)
        self.layer2 = self._make_layer(64, 128, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, num_blocks=2, stride=2)
        
        # Classification head
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(256, num_classes)
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = [ResidualBlock(in_channels, out_channels, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels, stride=1))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


# Create and test our ResNet
resnet = SimpleResNet(num_classes=10)
x = torch.randn(2, 3, 32, 32)
output = resnet(x)

total_params = sum(p.numel() for p in resnet.parameters())
print(f'SimpleResNet Architecture:')
print(f'  Input: {x.shape}')
print(f'  Output: {output.shape}')
print(f'  Total parameters: {total_params:,}')
print(f'\nKey insight: Skip connections allow gradients to flow directly,')
print(f'enabling training of very deep networks without degradation.')

In [ ]:
# Visualize: Why skip connections help gradient flow

# Compare gradient magnitudes in plain vs residual networks
class PlainBlock(nn.Module):
    """Plain block WITHOUT skip connection."""
    def __init__(self, channels=64):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return x

class ResBlock(nn.Module):
    """Residual block WITH skip connection."""
    def __init__(self, channels=64):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
    
    def forward(self, x):
        residual = F.relu(self.conv1(x))
        residual = self.conv2(residual)
        return F.relu(residual + x)  # Skip connection

# Build deep networks
num_blocks = 20
plain_net = nn.Sequential(*[PlainBlock(64) for _ in range(num_blocks)])
res_net = nn.Sequential(*[ResBlock(64) for _ in range(num_blocks)])

# Forward pass and compute gradients
x = torch.randn(1, 64, 8, 8, requires_grad=True)

# Plain network
out_plain = plain_net(x.clone().detach().requires_grad_(True))
loss_plain = out_plain.sum()
loss_plain.backward()

# Residual network
x_res = x.clone().detach().requires_grad_(True)
out_res = res_net(x_res)
loss_res = out_res.sum()
loss_res.backward()

# Compare gradient norms at different layers
plain_grads = [p.grad.norm().item() for p in plain_net.parameters() if p.grad is not None]
res_grads = [p.grad.norm().item() for p in res_net.parameters() if p.grad is not None]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(plain_grads, 'r-', alpha=0.7, label='Plain Network')
ax1.plot(res_grads, 'b-', alpha=0.7, label='Residual Network')
ax1.set_xlabel('Layer (parameter index)')
ax1.set_ylabel('Gradient Norm')
ax1.set_title('Gradient Magnitude Across Layers')
ax1.legend()
ax1.set_yscale('log')

ax2.bar(['Plain Net', 'ResNet'], 
        [np.mean(plain_grads), np.mean(res_grads)],
        color=['red', 'blue'], alpha=0.7)
ax2.set_ylabel('Mean Gradient Norm')
ax2.set_title('Average Gradient Magnitude')

plt.suptitle(f'Gradient Flow: {num_blocks} blocks deep', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Plain network mean gradient: {np.mean(plain_grads):.6f}')
print(f'ResNet mean gradient: {np.mean(res_grads):.6f}')
print(f'Ratio (ResNet/Plain): {np.mean(res_grads)/max(np.mean(plain_grads), 1e-10):.1f}x')

---
## 4. Vision Transformer (ViT)

### Key Idea (Dosovitskiy et al., 2020)

Apply the standard **Transformer encoder** (from NLP) directly to images:
1. Split image into fixed-size **patches** (e.g., 16×16 pixels)
2. Flatten each patch and project with a **linear embedding**
3. Add **position embeddings** (learnable)
4. Prepend a **[CLS] token** for classification
5. Process through standard Transformer encoder
6. Use [CLS] token output for classification

### Architecture

```
Image (224×224×3)
       │
       ▼
Split into patches (14×14 = 196 patches of 16×16×3)
       │
       ▼
Linear Projection: each patch → D-dimensional embedding
       │
       ▼
[CLS] + Patch Embeddings + Position Embeddings
       │
       ▼
┌─────────────────────────────────┐
│     Transformer Encoder         │ × L layers
│  ┌───────────────────────────┐  │
│  │ Multi-Head Self-Attention │  │
│  └───────────────────────────┘  │
│  ┌───────────────────────────┐  │
│  │    Feed-Forward Network   │  │
│  └───────────────────────────┘  │
│  (+ LayerNorm + Residuals)      │
└─────────────────────────────────┘
       │
       ▼
[CLS] token output → MLP Head → Class prediction
```

### ViT Variants

| Model | Layers | Hidden Dim | Heads | Params |
|-------|--------|-----------|-------|--------|
| ViT-Base | 12 | 768 | 12 | 86M |
| ViT-Large | 24 | 1024 | 16 | 307M |
| ViT-Huge | 32 | 1280 | 16 | 632M |

In [ ]:
# 4. Vision Transformer (ViT) - Implementation from Scratch

class PatchEmbedding(nn.Module):
    """Split image into patches and embed them."""
    def __init__(self, image_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        
        # Linear projection of flattened patches (equivalent to Conv2d)
        self.projection = nn.Conv2d(
            in_channels, embed_dim, 
            kernel_size=patch_size, stride=patch_size
        )
    
    def forward(self, x):
        # x: [B, C, H, W]
        x = self.projection(x)  # [B, embed_dim, H/P, W/P]
        x = x.flatten(2)        # [B, embed_dim, num_patches]
        x = x.transpose(1, 2)   # [B, num_patches, embed_dim]
        return x


class VisionTransformer(nn.Module):
    """Vision Transformer (ViT) for image classification."""
    def __init__(self, image_size=224, patch_size=16, in_channels=3,
                 num_classes=1000, embed_dim=768, num_heads=12,
                 num_layers=12, mlp_ratio=4.0):
        super().__init__()
        
        # Patch embedding
        self.patch_embed = PatchEmbedding(image_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches
        
        # [CLS] token: learnable classification token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Position embeddings (learnable)
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=int(embed_dim * mlp_ratio),
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-norm (as in ViT)
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Layer norm
        self.norm = nn.LayerNorm(embed_dim)
        
        # Classification head
        self.head = nn.Linear(embed_dim, num_classes)
        
        # Initialize position embeddings
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        
        # 1. Patch embedding
        x = self.patch_embed(x)  # [B, num_patches, embed_dim]
        
        # 2. Prepend [CLS] token
        cls_tokens = self.cls_token.expand(B, -1, -1)  # [B, 1, embed_dim]
        x = torch.cat([cls_tokens, x], dim=1)  # [B, num_patches+1, embed_dim]
        
        # 3. Add position embeddings
        x = x + self.pos_embed
        
        # 4. Transformer encoder
        x = self.encoder(x)
        
        # 5. Extract [CLS] token output
        x = self.norm(x[:, 0])  # [B, embed_dim]
        
        # 6. Classification
        x = self.head(x)  # [B, num_classes]
        return x


# Create a small ViT for demonstration
vit = VisionTransformer(
    image_size=32,      # Small images for demo
    patch_size=8,       # 4x4 = 16 patches
    in_channels=3,
    num_classes=10,
    embed_dim=128,      # Smaller for demo
    num_heads=4,
    num_layers=4,
    mlp_ratio=4.0
)

# Test forward pass
x = torch.randn(2, 3, 32, 32)
output = vit(x)

total_params = sum(p.numel() for p in vit.parameters())
print('Vision Transformer (ViT) - Demo Configuration:')
print(f'  Image size: 32×32')
print(f'  Patch size: 8×8')
print(f'  Number of patches: {(32//8)**2}')
print(f'  Sequence length: {(32//8)**2 + 1} (patches + [CLS] token)')
print(f'  Embedding dim: 128')
print(f'  Transformer layers: 4')
print(f'  Attention heads: 4')
print(f'  Output shape: {output.shape}')
print(f'  Total parameters: {total_params:,}')

In [ ]:
# Visualize ViT patch embedding process
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Create a sample image
np.random.seed(42)
sample_img = np.random.rand(32, 32, 3) * 0.5 + 0.25
# Add some structure
sample_img[8:24, 8:24, 0] = 0.8  # Red square in center
sample_img[12:20, 12:20, 1] = 0.9  # Green inner square

# 1. Original image
axes[0].imshow(sample_img)
axes[0].set_title('1. Original Image (32×32)')
axes[0].axis('off')

# 2. Image with patch grid
axes[1].imshow(sample_img)
patch_size = 8
for i in range(0, 33, patch_size):
    axes[1].axhline(y=i-0.5, color='white', linewidth=2)
    axes[1].axvline(x=i-0.5, color='white', linewidth=2)
# Number the patches
for i in range(4):
    for j in range(4):
        axes[1].text(j*8+3, i*8+5, str(i*4+j+1), color='white', 
                    fontsize=9, fontweight='bold')
axes[1].set_title('2. Split into 16 patches (8×8 each)')
axes[1].axis('off')

# 3. Sequence representation
seq_viz = np.random.rand(1, 17) * 0.7 + 0.15  # 1 CLS + 16 patches
seq_viz[0, 0] = 0.95  # CLS token highlighted
axes[2].imshow(seq_viz, aspect='auto', cmap='Blues')
axes[2].set_yticks([])
axes[2].set_xticks(range(17))
labels = ['[CLS]'] + [f'P{i+1}' for i in range(16)]
axes[2].set_xticklabels(labels, rotation=45, fontsize=8)
axes[2].set_title('3. Token Sequence (input to Transformer)')

plt.suptitle('ViT: Image → Patches → Token Sequence', fontsize=13)
plt.tight_layout()
plt.show()

print('Each patch (8×8×3 = 192 values) is linearly projected to embedding dim (128).')
print('Position embeddings are added to encode spatial information.')
print('[CLS] token aggregates information for classification.')

---
## 5. BEiT: BERT Pre-training for Images

### Key Idea (Bao et al., 2021)

Apply **BERT-style masked prediction** to images using **visual tokens**:

1. **Image Tokenizer** (discrete VAE): Converts image patches into discrete visual tokens
   (like words in a vocabulary)
2. **Masked Image Modeling**: Mask patches, predict their visual token IDs

### Difference from MAE

| | MAE | BEiT |
|---|-----|------|
| Target | Raw pixels | Discrete visual tokens |
| Loss | MSE (pixel reconstruction) | Cross-entropy (token prediction) |
| Tokenizer | Not needed | Discrete VAE (dVAE) |

### Architecture

```
Image ──┬──▶ Image Tokenizer (dVAE) ──▶ Visual Token IDs (targets)
        │
        └──▶ Patch Embedding ──▶ Mask 40% ──▶ ViT Encoder ──▶ Predict Token IDs
```

### Why Visual Tokens?

- Pixel-level prediction is too low-level (color, texture)
- Visual tokens capture **higher-level semantics** (object parts, shapes)
- Analogous to word tokens in NLP — discrete, meaningful units
- Cross-entropy loss on tokens works better than MSE on pixels

In [ ]:
# 5. BEiT: BERT Pre-training for Images (Simplified)

class SimpleImageTokenizer(nn.Module):
    """Simplified discrete VAE tokenizer.
    Maps image patches to discrete token IDs from a learned codebook.
    """
    def __init__(self, patch_dim=192, vocab_size=8192, embed_dim=256):
        super().__init__()
        self.vocab_size = vocab_size
        
        # Encoder: patch → continuous embedding
        self.encoder = nn.Sequential(
            nn.Linear(patch_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim)
        )
        
        # Codebook: discrete visual vocabulary
        self.codebook = nn.Embedding(vocab_size, embed_dim)
    
    def forward(self, patches):
        """Convert patches to discrete token IDs."""
        # Encode patches
        z = self.encoder(patches)  # [B, N, embed_dim]
        
        # Find nearest codebook entry (vector quantization)
        # Compute distances to all codebook entries
        codebook_weights = self.codebook.weight  # [vocab_size, embed_dim]
        distances = torch.cdist(z, codebook_weights.unsqueeze(0).expand(z.shape[0], -1, -1))
        
        # Get nearest token ID
        token_ids = distances.argmin(dim=-1)  # [B, N]
        return token_ids


class BEiTPretraining(nn.Module):
    """BEiT: Masked Image Modeling with visual tokens."""
    def __init__(self, image_size=32, patch_size=8, embed_dim=128,
                 num_heads=4, num_layers=4, vocab_size=8192, mask_ratio=0.4):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        self.mask_ratio = mask_ratio
        self.vocab_size = vocab_size
        patch_dim = 3 * patch_size * patch_size
        
        # Image tokenizer (frozen during BEiT training)
        self.tokenizer = SimpleImageTokenizer(patch_dim, vocab_size, embed_dim)
        
        # Patch embedding for ViT encoder
        self.patch_embed = nn.Linear(patch_dim, embed_dim)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
        self.mask_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim*4, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Prediction head: predict visual token ID
        self.head = nn.Linear(embed_dim, vocab_size)
    
    def patchify(self, images):
        B, C, H, W = images.shape
        p = self.patch_size
        patches = images.unfold(2, p, p).unfold(3, p, p)
        patches = patches.contiguous().view(B, C, -1, p, p)
        patches = patches.permute(0, 2, 1, 3, 4).contiguous()
        patches = patches.view(B, -1, C * p * p)
        return patches
    
    def forward(self, images):
        B = images.shape[0]
        patches = self.patchify(images)  # [B, N, patch_dim]
        
        # Get target token IDs from tokenizer (no gradient)
        with torch.no_grad():
            target_tokens = self.tokenizer(patches)  # [B, N]
        
        # Create mask
        num_masked = int(self.num_patches * self.mask_ratio)
        noise = torch.rand(B, self.num_patches)
        mask_indices = noise.argsort(dim=1)[:, :num_masked]
        
        # Embed patches
        x = self.patch_embed(patches) + self.pos_embed
        
        # Replace masked positions with mask token
        mask = torch.zeros(B, self.num_patches, dtype=torch.bool)
        mask.scatter_(1, mask_indices, True)
        x[mask] = self.mask_token.squeeze()
        
        # Encode
        x = self.encoder(x)
        
        # Predict tokens at masked positions
        masked_output = x[mask]  # [num_masked_total, embed_dim]
        logits = self.head(masked_output)  # [num_masked_total, vocab_size]
        
        # Target: token IDs at masked positions
        target = target_tokens[mask]  # [num_masked_total]
        
        # Cross-entropy loss
        loss = F.cross_entropy(logits, target)
        return loss, logits, target


# Demo
beit = BEiTPretraining(image_size=32, patch_size=8, embed_dim=128, mask_ratio=0.4)
images = torch.randn(4, 3, 32, 32)

loss, logits, targets = beit(images)

print('BEiT Pre-training Demo:')
print(f'  Image size: 32×32, Patch size: 8×8')
print(f'  Total patches: {beit.num_patches}')
print(f'  Masked patches: {int(beit.num_patches * 0.4)} (40%)')
print(f'  Visual vocabulary size: {beit.vocab_size}')
print(f'  Prediction logits shape: {logits.shape}')
print(f'  Target token IDs shape: {targets.shape}')
print(f'  Cross-entropy loss: {loss.item():.4f}')
print(f'\nKey: Predicting discrete tokens (not pixels) captures higher-level semantics.')

---
## 6. CLIP: Contrastive Language-Image Pre-training

### Key Idea (Radford et al., 2021)

Learn visual representations by training on **400 million (image, text) pairs** from the internet.
The model learns to match images with their text descriptions using contrastive learning.

### Architecture

```
Image ──▶ Image Encoder (ViT or ResNet) ──▶ Image Embedding ──┐
                                                                │ Contrastive
Text  ──▶ Text Encoder (Transformer)   ──▶ Text Embedding  ──┘ Loss
```

### Training Objective

Given a batch of N (image, text) pairs:
- **Positive pairs:** matching image-text (diagonal of similarity matrix)
- **Negative pairs:** non-matching image-text (off-diagonal)

Maximize similarity of N correct pairs while minimizing similarity of N²-N incorrect pairs.

### Zero-shot Classification

CLIP enables **zero-shot classification** without any task-specific training:
1. Create text prompts: "a photo of a {class_name}" for each class
2. Encode all prompts with text encoder
3. Encode the image with image encoder
4. Predict class = argmax of cosine similarities

```
                    "a photo of a dog"  ──▶ Text Encoder ──▶ t₁
                    "a photo of a cat"  ──▶ Text Encoder ──▶ t₂
Image ──▶ Image    "a photo of a bird" ──▶ Text Encoder ──▶ t₃
          Encoder
            │
            ▼
          img_emb ──▶ cos_sim(img_emb, t₁) = 0.92  ← Predicted: dog!
                      cos_sim(img_emb, t₂) = 0.31
                      cos_sim(img_emb, t₃) = 0.15
```

In [ ]:
# 6. CLIP: Contrastive Language-Image Pre-training (Simplified)

class CLIPModel(nn.Module):
    """Simplified CLIP: learns to match images with text descriptions."""
    def __init__(self, image_embed_dim=512, text_embed_dim=512, 
                 projection_dim=256, vocab_size=10000, max_seq_len=32):
        super().__init__()
        
        # Image Encoder (simplified CNN)
        self.image_encoder = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, image_embed_dim)
        )
        
        # Text Encoder (simplified Transformer)
        self.text_embedding = nn.Embedding(vocab_size, text_embed_dim)
        self.text_pos_embed = nn.Embedding(max_seq_len, text_embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=text_embed_dim, nhead=8, 
            dim_feedforward=text_embed_dim*4, batch_first=True
        )
        self.text_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.text_projection_pre = nn.Linear(text_embed_dim, text_embed_dim)
        
        # Projection heads (map to shared space)
        self.image_projection = nn.Linear(image_embed_dim, projection_dim)
        self.text_projection = nn.Linear(text_embed_dim, projection_dim)
        
        # Learnable temperature
        self.temperature = nn.Parameter(torch.tensor(0.07))
    
    def encode_image(self, images):
        features = self.image_encoder(images)
        projected = self.image_projection(features)
        return F.normalize(projected, dim=-1)
    
    def encode_text(self, token_ids):
        B, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)
        x = self.text_embedding(token_ids) + self.text_pos_embed(positions)
        x = self.text_encoder(x)
        # Use [EOS] token (last token) as text representation
        x = x[:, -1, :]  # [B, text_embed_dim]
        x = self.text_projection_pre(x)
        projected = self.text_projection(x)
        return F.normalize(projected, dim=-1)
    
    def forward(self, images, token_ids):
        # Encode both modalities
        image_embeds = self.encode_image(images)  # [B, projection_dim]
        text_embeds = self.encode_text(token_ids)  # [B, projection_dim]
        
        # Compute similarity matrix
        logits = (image_embeds @ text_embeds.T) / self.temperature  # [B, B]
        
        # Symmetric contrastive loss
        labels = torch.arange(len(images), device=images.device)
        loss_i2t = F.cross_entropy(logits, labels)      # Image-to-text
        loss_t2i = F.cross_entropy(logits.T, labels)    # Text-to-image
        loss = (loss_i2t + loss_t2i) / 2
        
        return loss, logits


# Demo
clip_model = CLIPModel()

# Simulate a batch of image-text pairs
batch_size = 8
images = torch.randn(batch_size, 3, 32, 32)
text_tokens = torch.randint(0, 10000, (batch_size, 16))  # Random token IDs

loss, similarity_matrix = clip_model(images, text_tokens)

print('CLIP Model Demo:')
print(f'  Batch size: {batch_size} image-text pairs')
print(f'  Image embedding shape: {clip_model.encode_image(images).shape}')
print(f'  Text embedding shape: {clip_model.encode_text(text_tokens).shape}')
print(f'  Similarity matrix shape: {similarity_matrix.shape}')
print(f'  Contrastive loss: {loss.item():.4f}')
print(f'  Temperature: {clip_model.temperature.item():.4f}')
print(f'\nDiagonal = positive pairs (should be high similarity)')
print(f'Off-diagonal = negative pairs (should be low similarity)')

In [ ]:
# Visualize CLIP similarity matrix
with torch.no_grad():
    img_emb = clip_model.encode_image(images)
    txt_emb = clip_model.encode_text(text_tokens)
    sim_matrix = (img_emb @ txt_emb.T).numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Similarity matrix
im = ax1.imshow(sim_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax1.set_xlabel('Text')
ax1.set_ylabel('Image')
ax1.set_title('CLIP Similarity Matrix (Image × Text)')
ax1.set_xticks(range(batch_size))
ax1.set_yticks(range(batch_size))
ax1.set_xticklabels([f'T{i}' for i in range(batch_size)])
ax1.set_yticklabels([f'I{i}' for i in range(batch_size)])
plt.colorbar(im, ax=ax1, fraction=0.046)

# Highlight diagonal (positive pairs)
for i in range(batch_size):
    ax1.add_patch(plt.Rectangle((i-0.5, i-0.5), 1, 1, 
                                fill=False, edgecolor='green', linewidth=2))

# Zero-shot classification concept
class_names = ['airplane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse']
# Simulate zero-shot scores
np.random.seed(42)
scores = np.random.dirichlet(np.ones(8)) * 3
scores[5] = scores.max() + 0.5  # Make 'dog' the highest

ax2.barh(class_names, scores, color='steelblue', alpha=0.8)
ax2.set_xlabel('Cosine Similarity')
ax2.set_title('Zero-shot Classification\n("a photo of a {class}")')
ax2.axvline(x=scores[5], color='green', linestyle='--', alpha=0.5)
ax2.annotate('Predicted!', xy=(scores[5], 5), fontsize=10, color='green')

plt.tight_layout()
plt.show()

print('Left: Similarity matrix — diagonal (green) should have highest values.')
print('Right: Zero-shot classification — no training on these classes needed!')

---
## 7. Practical Example: Using Pretrained Models

Let's use pretrained models from `torchvision` to demonstrate the power of foundation models.
We'll:
1. Load a pretrained ResNet-18 and ViT-B/16
2. Extract features from sample images
3. Fine-tune on CIFAR-10 with minimal data

In [ ]:
# 7. Practical Example: Using Pretrained Models from torchvision

# Load pretrained ResNet-18
resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet18.eval()

print('=== Pretrained ResNet-18 ===')
print(f'Total parameters: {sum(p.numel() for p in resnet18.parameters()):,}')
print(f'Output classes (ImageNet): 1000')

# Extract features (remove classification head)
resnet18_features = nn.Sequential(*list(resnet18.children())[:-1])  # Remove fc layer

# Test with a sample image
sample_image = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    features = resnet18_features(sample_image)
print(f'Feature shape (before fc): {features.shape}')  # [1, 512, 1, 1]
print(f'Feature dimension: 512')
print()

In [ ]:
# Load pretrained ViT-B/16 (if available in your torchvision version)
try:
    vit_b16 = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    vit_b16.eval()
    
    print('=== Pretrained ViT-B/16 ===')
    print(f'Total parameters: {sum(p.numel() for p in vit_b16.parameters()):,}')
    print(f'Patch size: 16×16')
    print(f'Image size: 224×224')
    print(f'Number of patches: {(224//16)**2} = 196')
    print(f'Embedding dim: 768')
    print(f'Transformer layers: 12')
    print(f'Attention heads: 12')
    
    # Forward pass
    with torch.no_grad():
        output = vit_b16(sample_image)
    print(f'Output shape: {output.shape}')  # [1, 1000]
    print(f'Top-5 predicted classes: {output.topk(5).indices[0].tolist()}')
    
except Exception as e:
    print(f'ViT not available in this torchvision version: {e}')
    print('Requires torchvision >= 0.13')

In [ ]:
# Fine-tuning a pretrained ResNet-18 on CIFAR-10

def create_finetuned_model(num_classes=10, freeze_backbone=True):
    """Create a fine-tuned model from pretrained ResNet-18."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    
    # Optionally freeze backbone
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    
    # Replace classification head for new task
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    
    return model

# Create fine-tuned model
finetuned = create_finetuned_model(num_classes=10, freeze_backbone=True)

trainable = sum(p.numel() for p in finetuned.parameters() if p.requires_grad)
total = sum(p.numel() for p in finetuned.parameters())

print('=== Fine-tuning ResNet-18 for CIFAR-10 ===')
print(f'Total parameters: {total:,}')
print(f'Trainable parameters: {trainable:,}')
print(f'Frozen parameters: {total - trainable:,}')
print(f'Training only {trainable/total*100:.2f}% of parameters!')
print()

# Quick training loop demo
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print('Transform pipeline for pretrained models:')
print('  1. Resize to 224×224 (ImageNet input size)')
print('  2. ToTensor (scale to [0, 1])')
print('  3. Normalize with ImageNet mean/std')
print()

# Training setup
optimizer = optim.Adam(finetuned.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Simulate one training step
finetuned.train()
dummy_batch = torch.randn(4, 3, 224, 224)
dummy_labels = torch.randint(0, 10, (4,))

output = finetuned(dummy_batch)
loss = criterion(output, dummy_labels)
loss.backward()
optimizer.step()

print(f'Training step completed!')
print(f'  Batch output shape: {output.shape}')
print(f'  Loss: {loss.item():.4f}')
print(f'\nWith pretrained features, CIFAR-10 accuracy > 90% with just a few epochs!')

In [ ]:
# Visualize: Feature comparison between layers of pretrained ResNet

# Hook to capture intermediate features
features_dict = {}

def get_hook(name):
    def hook(model, input, output):
        features_dict[name] = output.detach()
    return hook

# Register hooks on different layers
resnet18_viz = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
resnet18_viz.eval()

resnet18_viz.layer1.register_forward_hook(get_hook('layer1'))
resnet18_viz.layer2.register_forward_hook(get_hook('layer2'))
resnet18_viz.layer3.register_forward_hook(get_hook('layer3'))
resnet18_viz.layer4.register_forward_hook(get_hook('layer4'))

# Forward pass
with torch.no_grad():
    _ = resnet18_viz(torch.randn(1, 3, 224, 224))

# Visualize feature maps at different depths
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

layer_names = ['Layer 1\n(edges, textures)', 'Layer 2\n(patterns, shapes)', 
               'Layer 3\n(object parts)', 'Layer 4\n(high-level semantics)']

for idx, (name, feat) in enumerate(features_dict.items()):
    # Show first feature map
    axes[0, idx].imshow(feat[0, 0].numpy(), cmap='viridis')
    axes[0, idx].set_title(f'{layer_names[idx]}\nShape: {list(feat.shape[1:])}')
    axes[0, idx].axis('off')
    
    # Show feature map statistics
    feat_flat = feat[0].flatten(1)
    axes[1, idx].hist(feat_flat.mean(1).numpy(), bins=20, alpha=0.7, color='steelblue')
    axes[1, idx].set_title(f'Channel activations\n(mean per channel)')
    axes[1, idx].set_xlabel('Activation')

plt.suptitle('Pretrained ResNet-18: Feature Hierarchy', fontsize=14)
plt.tight_layout()
plt.show()

print('Early layers → low-level features (edges, textures)')
print('Deep layers → high-level features (object parts, semantics)')
print('These general features transfer well to new tasks!')

---
## 8. Summary

### Key Takeaways

| Concept | Key Idea |
|---------|----------|
| **Pre-train + Fine-tune** | Learn general features from large unlabeled data, adapt with small labeled data |
| **Self-supervised Learning** | Create supervision from data structure itself (no labels needed) |
| **Colorization** | Predict colors from grayscale → learns object semantics |
| **Contrastive Learning** | Pull augmented views together, push different images apart (InfoNCE) |
| **Masked Image Modeling** | Mask 75% patches, reconstruct → learns holistic understanding |
| **Rotation Prediction** | Predict rotation angle → learns spatial/orientation features |
| **Jigsaw Puzzle** | Predict patch order → learns spatial relationships |
| **ResNet** | Skip connections H(x)=F(x)+x solve degradation, enable deep networks |
| **ViT** | Patches → tokens → Transformer encoder; [CLS] for classification |
| **BEiT** | BERT-style: mask patches, predict discrete visual tokens (not pixels) |
| **CLIP** | Match images with text via contrastive learning → zero-shot classification |

### The Evolution

```
Supervised (ImageNet)  →  Self-supervised (SimCLR, MAE)  →  Multi-modal (CLIP)
     ResNet                      ViT + BEiT                    ViT + Text
  Labeled data needed        No labels needed            Image-text pairs
  Task-specific              General features            Zero-shot capable
```

### Practical Guidelines

1. **Start with pretrained models** — almost always better than training from scratch
2. **Linear probing first** — freeze backbone, train only the head
3. **Full fine-tuning if needed** — unfreeze with small learning rate
4. **Use appropriate transforms** — match the pre-training preprocessing
5. **Consider CLIP for zero-shot** — no task-specific training needed

### References

- He et al. (2015) - Deep Residual Learning for Image Recognition
- Dosovitskiy et al. (2020) - An Image is Worth 16x16 Words (ViT)
- Chen et al. (2020) - A Simple Framework for Contrastive Learning (SimCLR)
- Bao et al. (2021) - BEiT: BERT Pre-Training of Image Transformers
- Radford et al. (2021) - Learning Transferable Visual Models (CLIP)
- He et al. (2022) - Masked Autoencoders Are Scalable Vision Learners (MAE)

In [ ]:
# Summary: Model comparison
print('=' * 70)
print('FOUNDATION MODELS IN COMPUTER VISION - COMPARISON')
print('=' * 70)
print(f'{"Model":<12} {"Type":<15} {"Pre-training":<25} {"Key Innovation"}')
print('-' * 70)
print(f'{"ResNet":<12} {"CNN":<15} {"Supervised (ImageNet)":<25} {"Skip connections"}')
print(f'{"SimCLR":<12} {"CNN":<15} {"Contrastive (SSL)":<25} {"Augmentation pairs"}')
print(f'{"ViT":<12} {"Transformer":<15} {"Supervised (ImageNet)":<25} {"Patches as tokens"}')
print(f'{"BEiT":<12} {"Transformer":<15} {"Masked modeling (SSL)":<25} {"Visual token prediction"}')
print(f'{"MAE":<12} {"Transformer":<15} {"Masked modeling (SSL)":<25} {"75% masking + pixel recon"}')
print(f'{"CLIP":<12} {"Multi-modal":<15} {"Contrastive (img+text)":<25} {"Zero-shot transfer"}')
print('=' * 70)
print()
print('Key insight: Self-supervised pre-training on large data produces')
print('foundation models that transfer effectively to downstream tasks')
print('with minimal labeled data.')